In [0]:
import pyspark.sql.functions as F

T_cust = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter(F.col("snapshot_date_id") == "20260330")
    # .filter(F.col("renewal_status").isin("Active", "Active_Non_Renewing"))
    # .filter(F.col("plan_id").isin("annual1", "gift-plan"))
    .select("cde_id", "plan_id", "renewal_status")
)

T_trans = (
    spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
    .filter(F.col("VisitDateId") > "20250101")
    .select("CDE_ID", "VisitDateId", "LocationId")
)

T_loc = (
    spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
    .filter(F.col("Location_Name").like("%Miramichi%"))
    .select("LocationID", "Location_Name")
)

df = (
    T_trans.alias("T")
    .join(T_loc.alias("L"), F.col("T.LocationId") == F.col("L.LocationID"), "inner") 
    .join(T_cust.alias("C"), F.col("T.CDE_ID") == F.col("C.cde_id"), "inner")        
    .select(
        F.col("T.CDE_ID"),
        F.col("C.plan_id"),
        F.col("C.renewal_status"),
        F.col("L.Location_Name"),
        F.col("T.VisitDateId")
    )
)

In [0]:
display(df)

In [0]:
T_subs = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")

df_end_2025 = (
  df.select("CDE_ID")
  .join(T_subs, T_subs.cde_id == df.CDE_ID, "inner")
  .filter(F.col("snapshot_date_id") == "20251231")
  .select(df.CDE_ID, "plan_id", "renewal_status")
)

In [0]:
df_end_2025.display()

In [0]:
df_pivot = (
    df_end_2025
    .groupBy("plan_id")
    .pivot("renewal_status")
    .agg(F.countDistinct("cde_id").alias("distinct_cde"))
)

df_pivot.display()

In [0]:


display(df.filter(F.col("plan_id") == "gift-plan").agg(F.countDistinct("CDE_ID")))


In [0]:
import pyspark.sql.functions as F

df_pivot = (
    df
    .withColumn("category", F.concat_ws(" | ", F.col("renewal_status"), F.col("plan_id")))
    .groupBy()
    .pivot(
        "category",
        [
            "Active | annual1",
            "Active_Non_Renewing | annual1",
            "Active | gift-plan",
            "Active_Non_Renewing | gift-plan"
        ]
    )
    .agg(F.countDistinct("CDE_ID"))
)

In [0]:
display(df_pivot)

In [0]:
%sql
SELECT
    T.CDE_ID,
    C.plan_id,
    C.renewal_status,
    L.Location_Name,
    T.VisitDateId
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction T

INNER JOIN (
    SELECT cde_id, plan_id, renewal_status
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = '20260330'
      AND renewal_status IN ('Active', 'Active_Non_Renewing')
      AND plan_id IN ('annual1', 'gift-plan')
) C
ON T.CDE_ID = C.cde_id

INNER JOIN (
    SELECT LocationID, Location_Name
    FROM cpx_dataanalytics_gold.customer360.cpx_d_location
    WHERE Location_Name LIKE '%Miramichi%'
) L
ON T.LocationId = L.LocationID

WHERE T.VisitDateId > '20250101'